In [0]:
# Definir rutas
staging_path = "/Volumes/iniciacion_deportiva/bronze/staging_zone"
checkpoint_path = "/Volumes/iniciacion_deportiva/bronze/checkpoints/ventas_raw"
# Configurar Auto Loader para leer desde el Volume
df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet") # O "parquet" según lo que subas
    .option("cloudFiles.schemaLocation", checkpoint_path) # Detecta cambios en el archivo
    .option("header", "true")
    .load(staging_path))

# Añadir metadatos de auditoría
from pyspark.sql.functions import current_timestamp, lit

df_final = df_raw.select(
    "*", 
    lit("mercadolibre_api").alias("_source"), 
    current_timestamp().alias("ingested_at")
)

# Escribir en la tabla Bronze de forma incremental
(df_final.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .outputMode("append")
    .toTable("iniciacion_deportiva.bronze.ventas_raw"))